# Modal

In [ ]:
# | default_exp run_modal

In [ ]:
# | hide
from nbdev.showdoc import *  # type: ignore

In [1]:
import os
import subprocess
import sys
from typing import TypeAlias, List, Tuple
from dotenv import load_dotenv # type: ignore

if os.getenv("USER") == "max":
    # Get the current working directory
    current_dir = os.getcwd()

    # Append the parent directory to sys.path
    parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
    sys.path.append(parent_dir)

    # Decrypt the .env.local file and load variables
    try:
        # Decrypt and output to stdout
        os.system(f"dotenvx decrypt -f {os.path.abspath(os.path.join(current_dir, '..'))}/.env.local")

        # Load the decrypted environment variables from stdout
        load_dotenv(dotenv_path=os.path.abspath(os.path.join(current_dir, '..')) + "/.env.local")

        os.system(f"dotenvx encrypt -f {os.path.abspath(os.path.join(current_dir, '..'))}/.env.local")

        print("Environment variables loaded successfully.")
        os.environ["DOCKER_HOST"] = "unix:///Users/max/.docker/run/docker.sock"
    except subprocess.CalledProcessError as e:
        print(f"Error decrypting .env.local: {e}")
        sys.exit(1)

no changes (/Users/max/Documents/product_horse/.env.local)
✔ encrypted (/Users/max/Documents/product_horse/.env.local)
Environment variables loaded successfully.


In [2]:
# | export
import modal
from product_horse.audio_video import create_video_from_utterances
from product_horse.db import SqlModelDatabase, UtteranceSegment
from product_horse.filesystems import LocalFileSystem,R2StorageClient  
from uuid import UUID
from typing import TypedDict, List
from random import randint


In [47]:
# | export
app = modal.App('process-video')

class UtteranceSegmentInput(TypedDict):
    utterance_id: str
    word_ids: List[str]


image = (
    modal.Image.debian_slim()
    .apt_install("ffmpeg")
    .pip_install("uv")
    .run_commands("uv pip install requirements.txt")
)

@app.function( # type: ignore
    gpu="A10G",
    secrets=[modal.Secret.from_name("video_secrets")]
)
async def run_remote_create_video(utterance_segments_inputs: list[UtteranceSegmentInput], employee_id: UUID, jwt: str, title: str):
    db_url = os.getenv("DATABASE_URL")
    API_URL = os.getenv("API_URL")
    if db_url is None:
        raise Exception("DATABASE_URL is not set")
    database = SqlModelDatabase(
        database_url=db_url
    )
    employee = database.get_employee(employee_id=employee_id)
    if employee is None:
        raise Exception("Employee not found")
    r2 = R2StorageClient(
        api_url=API_URL,
        base_path=str(employee.company_id),
        jwt=jwt,
    )
    server_file_system = LocalFileSystem()
    utterance_ids = [UUID(segment["utterance_id"]) for segment in utterance_segments_inputs]
    word_ids = [word_id for segment in utterance_segments_inputs for word_id in segment["word_ids"]]
    utterances = database.as_employee(employee).get_utterances(
        utterance_ids, word_ids=word_ids
    )
    video_exists = database.as_employee(employee).get_all_videos()
    for video in video_exists:
        if video.title == title: # type: ignore
            title = f"{title}-{randint(1, 200)}"
    utterance_segments_for_video : list[UtteranceSegment] = []
    for utterance in utterances:
        utterance_segments_for_video.append(
            UtteranceSegment(
                id=utterance.id,
                transcription_id=utterance.transcription_id,
                transcription=utterance.transcription,
                words=utterance.words,
                segment_start_word=utterance.words[0], # type: ignore
                segment_end_word=utterance.words[-1], # type: ignore
                confidence=utterance.confidence,
                company_id=utterance.company_id,
                created_by_id=utterance.created_by_id,
                start=utterance.start,
                end=utterance.end,
            )
        )
    user = database.as_employee(employee).get_all_users()[0]
    if user is None:
        raise Exception("User not found")
    final_destination = await r2.get_unique_file_key(f"{title}.mp4", str(user.id))
    video = await create_video_from_utterances(
        database,
        remote_file_system=r2,
        local_file_system=server_file_system,
        employee=employee,
        user=user,
        utterance_segments=utterance_segments_for_video,
        final_destination=final_destination,
        title=title,
        size=(1920, 1080),
        force_size=True,
    )


In [53]:
lookup_video = modal.Function.lookup("process-video", "list_files")

In [54]:
print(lookup_video.remote())

total 308319
drwxr-xr-x 2 root root         0 Aug 23 00:09 .
drwxr-xr-x 1 root root       172 Aug 23 00:09 ..
drwxr-xr-x 2 root root         0 Aug 23 00:09 0ac7b64f-27e6-4297-9c74-6ea08640ba38
drwxr-xr-x 2 root root         0 Aug 23 00:09 667dd149-ef7c-4b4c-ae6b-8375cea1ea48
drwxr-xr-x 2 root root         0 Aug 23 00:09 9d34e4af-18a2-4517-9c5d-f64c44e363ed
-rw-r--r-- 1 root root    931974 Aug  6 02:49 9ff6ed44-4a2b-41d8-a841-1b92f93ce11e
-rw-r--r-- 1 root root    931974 Aug  8 00:05 a0844e3b-07b9-4b53-96de-6a2db9c24499
-rw-r--r-- 1 root root    931974 Aug  8 00:07 a56854c5-a2b8-4a1f-bc76-76adae3462a2
-rw-r--r-- 1 root root    931974 Jul 25 16:06 aaf855e7-f69f-4266-9157-8e23b39f5a2c
drwxr-xr-x 2 root root         0 Aug 23 00:09 b28c5b28-2bd9-471f-9f48-7a8968573aa3
-rw-r--r-- 1 root root    931974 Aug  6 02:47 b696de4b-8995-4896-8ec5-80d78b910091
-rw-r--r-- 1 root root    931974 Jul 28 01:58 c4467e70-8c0b-46c0-b968-f21aa17ac85a
drwxr-xr-x 2 root root         0 Aug 23 00:09 cce02773-d237-

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore